# Data Augmentation in NLP Using Back Translation With Transformers:  MarianMT       


In [ ]:
# Back Translation using deep_translator
# Translate text to another language and back to the original.
    
from deep_translator import GoogleTranslator # pip install deep_translator

def back_translate(text, lang='fr'):
    translated = GoogleTranslator(source='auto', target=lang).translate(text)
    back_translated = GoogleTranslator(source=lang, target='en').translate(translated)
    return back_translated

original = "The human brain is capable of complex language processing, but decoding thoughts directly from brain activity has long been a challenge"
translate = back_translate(original) 
print(f"original:  {original}")
print(f"translate: {translate}")


original = "Imagine being able to translate thoughts into words without speaking or typing"
translate = back_translate(original) 
print(f"original:  {original}")
print(f"translate: {translate}")

# OUTPUT: 
# original:  The human brain is capable of complex language processing, but decoding thoughts directly from brain activity has long been a challenge
# translate: The human brain is capable of complex language treatment, but the decoding of thoughts directly from brain activity has long been a challenge
# original:  Imagine being able to translate thoughts into words without speaking or typing
# translate: Imagine being able to translate thoughts in words without speaking or typing


In [ ]:
# Using pre-trained model
!pip install transformers
!pip install sentencepiece
from transformers import MarianMTModel, MarianTokenizer

**Note**:
Make sure to restart the kernel so that the changes are taken into consideration.

# Models Configuration

## Configuration of the first model
This model translates from English to French

In [2]:
# Get the name of the first model
first_model_name = 'Helsinki-NLP/opus-mt-en-fr' # grants access

# Get the tokenizer
first_model_tkn = MarianTokenizer.from_pretrained(first_model_name)

# Load the pretrained model based on the name
first_model = MarianMTModel.from_pretrained(first_model_name)

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

## Configuration of the second model
This model translates from French to English

In [3]:
# Get the name of the second model
second_model_name = 'Helsinki-NLP/opus-mt-fr-en'

# Get the tokenizer
second_model_tkn = MarianTokenizer.from_pretrained(second_model_name)

# Load the pretrained model based on the name
second_model = MarianMTModel.from_pretrained(second_model_name)

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Below is the text that will be used for the experimentation.

In [30]:
original_texts = [
    "The technology could revolutionize how we interact with machines.",
    "By decoding neural activity, researchers aim to give voice to the voiceless.",
    "This breakthrough opens the door to non-verbal communication through AI."
]

['The technology could revolutionize how we interact with machines.',
 'By decoding neural activity, researchers aim to give voice to the voiceless.',
 'This breakthrough opens the door to non-verbal communication through AI.']

In [ ]:
print(original_texts)

In [31]:
def format_batch_texts(language_code, batch_texts):
    formated_bach = [">>{}<< {}".format(language_code, text) for text in batch_texts]
    return formated_bach



['>>fr<< The technology could revolutionize how we interact with machines.',
 '>>fr<< By decoding neural activity, researchers aim to give voice to the voiceless.',
 '>>fr<< This breakthrough opens the door to non-verbal communication through AI.']

In [40]:
# Test of the function
formatted_batch_text =  format_batch_texts("fr", original_texts)
print(formatted_batch_text)

['>>fr<< The technology could revolutionize how we interact with machines.', '>>fr<< By decoding neural activity, researchers aim to give voice to the voiceless.', '>>fr<< This breakthrough opens the door to non-verbal communication through AI.']


In [32]:
def perform_translation(batch_texts, model, tokenizer, to_language="fr"):
    # Prepare the text data into appropriate format for the model
    formated_batch_texts = format_batch_texts(to_language, batch_texts)

    # Generate translation using model
    translated = model.generate(**tokenizer(formated_batch_texts, return_tensors="pt", padding=True))

    # Convert the generated tokens indices back into text
    translated_texts = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

    return translated_texts

translated_texts = perform_translation(original_texts, first_model, first_model_tkn)
translated_texts

['La technologie pourrait révolutionner la façon dont nous interagissons avec les machines.',
 "En décodant l'activité neuronale, les chercheurs cherchent à donner la voix aux sans-voix.",
 "Cette percée ouvre la porte à la communication non verbale par l'IA."]

Now we can translate back the following texts into English by providing the proper parameters.

In [33]:
back_translated_texts = perform_translation(translated_texts, second_model, second_model_tkn)

In [34]:
back_translated_texts

['Technology could revolutionize the way we interact with machines.',
 'By decoding neuronal activity, researchers seek to give voice to voiceless people.',
 'This breakthrough opens the door to non-verbal communication by AI.']

We can notice that the three sentence in the original batch has changed slightly and retains the original meaning.

In [51]:
for i in range(len(original_texts)):
  print(f"original:   {original_texts[i]}")
  print(f"translated: {back_translated_texts[i]}")

original:   The technology could revolutionize how we interact with machines.
translated: Technology could revolutionize the way we interact with machines.
original:   By decoding neural activity, researchers aim to give voice to the voiceless.
translated: By decoding neuronal activity, researchers seek to give voice to voiceless people.
original:   This breakthrough opens the door to non-verbal communication through AI.
translated: This breakthrough opens the door to non-verbal communication by AI.
